# 3장. CAE 학습 데이터 구성과 전처리

## 개요

**학습 목표:** 제공 VTU 10개를 모델용 입력/정답 묶음으로 구성

| 장 구성 | 주요 내용 |
|---|---|
| 1. 입력 자료와 실행환경 | 입력 파일/준비 기록/실행 경로 |
| 2. 학습/검증/시험 케이스 분할 | 분할 기준/목록 검증/조건 분포 |
| 3. 정규화와 역정규화 | 계산 원리/학습 통계/실제 값 변환 |
| 4. 케이스별 노드 샘플링 | 선택 개수/학습과 평가의 선택 규칙 |
| 5. PyTorch Dataset 구현 | 반환 항목/처리 흐름/케이스 조회/선택 규칙 확인 |
| 6. DataLoader와 배치 구성 | 배치 크기/텐서 차원/실행 설정 |
| 7. 학습 자료 준비 상태 점검 | 공통 처리 규칙/배치의 입력과 정답 점검 |
| 8. 다음 장 준비와 오류 확인 | 준비 파일 유지/주요 오류 확인 |


## 1. 입력 자료와 실행환경


### 1.3 학습 자료 로딩과 준비 기록 확인


### 03장 / 셀 03 / Notebook


In [ ]:
# 1. 파일 경로, 숫자 배열, PyTorch 계산에 사용할 기본 도구를 준비합니다.
from pathlib import Path
import json
import sys
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 2. 현재 위치와 상위 폴더에서 교육용 데이터 처리 코드를 찾습니다.
working_dir = Path.cwd().resolve()
candidates = []
for parent in (working_dir, *working_dir.parents):
    candidates.extend([parent / "course_utils" / "dataset.py",
                       parent / "education" / "course_utils" / "dataset.py"])
module_path = next((path for path in candidates if path.is_file()), None)
if module_path is None:
    raise FileNotFoundError("프로젝트 루트 또는 education 하위 폴더에서 실행하세요.")
# 3. 찾은 교육 폴더를 연결하고 공통 데이터 처리/검사 도구를 가져옵니다.
education_dir = module_path.parent.parent
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))
from course_utils.dataset import (FIELDS, N_POINTS, SEED, HookPointDataset, fit_train_statistics,
                                 load_prepared, make_split, point_indices)
from course_utils.dataset_checks import audit_dataset
# 4. VTU/조건표와 준비 기록을 읽고, 파일/분할/통계가 같은 자료인지 확인합니다.
run_dir = education_dir / "runs" / "dataset"
cases, split, normalizer, fixed_indices, manifest = load_prepared(education_dir / "data", run_dir)
# 5. 로딩된 케이스 수, 분할/통계 식별값, 준비 기록 폴더를 출력합니다.
print("케이스 수:", len(cases))
print("분할 해시:", split["split_sha256"])
print("통계 해시:", normalizer.stats["stats_sha256"])
print("결과 폴더:", run_dir)


## 2. 학습/검증/시험 케이스 분할


### 2.4 케이스 분할 목록 확인


### 03장 / 셀 06 / Notebook


In [ ]:
# 1. 같은 규칙으로 분할을 다시 만들고 저장된 목록과 비교합니다.
regenerated = make_split(cases, seed=42)
assert regenerated == split
# 2. 케이스가 두 그룹에 겹치거나 어느 그룹에서도 빠지는 일이 없는지 검사합니다.
groups = [set(split[role]) for role in ["train", "validation", "test"]]
assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
assert set.union(*groups) == set(cases)
# 3. 학습/검증/시험의 케이스 수와 이름을 출력합니다.
for role in ["train", "validation", "test"]:
    print(role, len(split[role]), split[role])
# 4. 검증/시험 조건이 학습 조건의 볼록껍질 안에 있는지 저장된 확인 결과를 출력합니다.
print("검증/시험 조건이 학습 볼록껍질 내부:", split["holdout_inside_train_convex_hull"])


## 3. 정규화와 역정규화


### 3.5 응력 정규화/역정규화 확인 실습


### 03장 / 셀 11 / Notebook


In [ ]:
# 1. 설명용 노드 5개의 응력과 3.2절의 평균/표준편차를 준비합니다.
physical_stress = np.array([80, 100, 120, 140, 160], dtype=np.float64)
example_mean = 100.0  # 설명용 평균, 단위 MPa
example_std = 20.0    # 설명용 표준편차, 단위 MPa
# 2. 정규화: 평균을 빼고 표준편차로 나눕니다.
normalized_stress = (physical_stress - example_mean) / example_std
# 3. 역정규화: 같은 표준편차를 곱하고 평균을 더합니다.
restored_stress = normalized_stress * example_std + example_mean
# 4. 같은 노드의 정답/정규화/역정규화 값을 순서대로 출력합니다.
print("정답 응력(MPa):", physical_stress)
print("정규화 응력:", normalized_stress)
print("역정규화 응력(MPa):", restored_stress)
# 5. 처음 응력으로 돌아왔는지 검사합니다. 실제 학습 기준은 바꾸지 않습니다.
assert np.allclose(restored_stress, physical_stress, rtol=1e-12, atol=1e-12)


## 5. PyTorch Dataset 구현


### 5.2 Dataset의 처리 흐름과 코드


### 03장 / 셀 14 / 코드 조각 01


```python
case_id = split["train"][0]       # 학습 목록의 첫 케이스 이름
example_case = cases[case_id]    # 그 이름에 해당하는 해석 자료
indices = [0, 1, 2]             # 배열의 첫 세 행: ANSYS 노드 ID와는 다름
example_case.coords[indices]    # 선택한 세 행의 X/Y/Z 좌표
example_case.stress[indices]    # 같은 세 행의 응력 정답
```


### 5.3 노드별 좌표/정답 대응 관계


### 03장 / 셀 15 / Notebook


In [ ]:
# Dataset이 자료를 준비하고 반환하는 방법을 정의합니다.
class HookPointDataset(Dataset):
    """One item per case; dynamic across training epochs, fixed for holdouts."""
    # 준비: 사용할 케이스, 정규화 기준, 노드 선택 설정을 등록합니다.
    def __init__(self, cases, case_ids, normalizer, n_points=N_POINTS, seed=SEED,
                 dynamic=False, fixed_indices=None):
        self.cases = cases
        self.case_ids = tuple(case_ids)
        self.normalizer = normalizer
        self.n_points = n_points
        self.seed = seed
        self.dynamic = dynamic
        self.epoch = 0
        self.fixed_indices = {}
        # 준비 자료 확인: 케이스 목록과 고정 노드 선택 기록을 검사합니다.
        if len(self.case_ids) != len(set(self.case_ids)):
            raise ValueError("Duplicate dataset cases.")
        for case_id in self.case_ids:
            if case_id not in cases:
                raise ValueError(f"Unknown case: {case_id}")
            if not dynamic:
                selected = (point_indices(cases[case_id], n_points, seed) if fixed_indices is None
                            else np.array(fixed_indices[case_id], dtype=np.int64))
                if selected.shape != (n_points,) or np.unique(selected).size != n_points:
                    raise ValueError("Fixed sample has duplicate indices or wrong point count.")
                if selected.min() < 0 or selected.max() >= cases[case_id].node_ids.size:
                    raise ValueError("Fixed sample has out-of-range indices.")
                self.fixed_indices[case_id] = selected

    # 이 Dataset에서 꺼낼 수 있는 케이스 수를 알려 줍니다.
    def __len__(self):
        return len(self.case_ids)

    # 학습 반복 번호를 기록해 다음 노드 선택에 사용합니다.
    def set_epoch(self, epoch):
        if int(epoch) != epoch or epoch < 0:
            raise ValueError("Epoch must be a non-negative integer.")
        self.epoch = int(epoch)

    # 노드 선택: 학습은 현재 반복 번호에 맞춰 고르고, 평가는 저장된 선택을 사용합니다.
    def sample_indices(self, index):
        case_id = self.case_ids[index]
        if self.dynamic:
            return point_indices(self.cases[case_id], self.n_points, self.seed, self.epoch)
        return self.fixed_indices[case_id].copy()

    def __getitem__(self, index):
        # 1. 요청받은 케이스를 선택합니다.
        case_id = self.case_ids[index]
        case = self.cases[case_id]
        # 2. 선택한 케이스에서 이번에 사용할 노드 위치 목록을 얻습니다.
        indices = self.sample_indices(index)
        # 3. 입력과 정답 준비: 같은 노드의 좌표/응력/변위를 꺼내고 두께/하중을 연결합니다.
        raw = {"coords": case.coords[indices], "globals": case.globals,
               "stress": case.stress[indices], "displacement": case.displacement[indices]}
        # 4. 정규화: 저장된 학습 기준을 적용하고 PyTorch 계산용 숫자 묶음으로 바꿉니다.
        item = {field: torch.from_numpy(self.normalizer.normalize(field, values).astype(np.float32))
                for field, values in raw.items()}
        # 5. 원래 해석 자료를 찾을 수 있도록 노드 번호와 케이스 이름을 붙입니다.
        item["node_ids"] = torch.from_numpy(case.node_ids[indices].copy())
        item["case_id"] = case_id
        # 6. 준비된 입력/정답/확인 정보를 한 케이스의 묶음으로 반환합니다.
        return item


### 5.4 Dataset 생성과 케이스 조회


### 03장 / 셀 17 / Notebook


In [ ]:
# 1. 학습/검증/시험 목록과 각각의 노드 선택 규칙으로 Dataset 세 개를 준비합니다.
datasets = {
    role: HookPointDataset(cases, split[role], normalizer, n_points=4096, seed=42,
                           dynamic=(role == "train"), fixed_indices=fixed_indices)
    for role in ["train", "validation", "test"]
}

# 2. 학습 Dataset의 첫 번째 케이스에서 입력/정답 묶음을 꺼냅니다.
item = datasets["train"][0]
# 3. 어느 케이스인지, 반환된 자료의 배열 모양과 자료형은 무엇인지 확인합니다.
print("케이스:", item["case_id"])
for field in ["coords", "globals", "stress", "displacement", "node_ids"]:
    print(field, tuple(item[field].shape), item[field].dtype)

# 4. 주요 항목의 배열 모양이 설정과 맞는지 검사합니다. 통과하면 추가 출력 없이 끝납니다.
assert item["coords"].shape == (4096, 3)
assert item["globals"].shape == (2,)
assert item["stress"].shape == (4096, 1)


### 5.5 노드 선택 규칙 확인 실습


### 03장 / 셀 19 / Notebook


In [ ]:
# 1. 학습 Dataset을 선택하고 반복 번호를 0으로 맞춥니다.
train_dataset = datasets["train"]
train_dataset.set_epoch(0)
# 2. 같은 학습 케이스를 두 번 요청해 선택된 노드가 같은지 검사합니다.
epoch_zero = train_dataset[0]["node_ids"].clone()
assert torch.equal(epoch_zero, train_dataset[0]["node_ids"])
# 3. 반복 번호를 1로 바꾸고, 학습 노드 선택이 달라지는지 검사합니다.
train_dataset.set_epoch(1)
epoch_one = train_dataset[0]["node_ids"].clone()
assert not torch.equal(epoch_zero, epoch_one)
# 4. 다음 배치 예제를 위해 학습 반복 번호를 0으로 되돌립니다.
train_dataset.set_epoch(0)

# 5. 검증 Dataset은 반복 번호를 바꿔도 같은 노드를 반환하는지 검사합니다.
validation_dataset = datasets["validation"]
validation_zero = validation_dataset[0]["node_ids"].clone()
validation_dataset.set_epoch(1)
assert torch.equal(validation_zero, validation_dataset[0]["node_ids"])
# 6. 검사 결과와 학습 묶음의 고유 노드 수를 출력합니다.
print("학습: 같은 에폭 재현, 다음 에폭 변경")
print("검증: 에폭 변경 후에도 동일한 노드")
print("현재 학습 샘플 고유 노드 수:", int(epoch_zero.unique().numel()))


## 6. DataLoader와 배치 구성


### 6.3 배치 구성 확인 실습


### 03장 / 셀 22 / Notebook


In [ ]:
# 1. 학습/검증/시험 Dataset을 케이스 2개씩 묶어 전달하도록 준비합니다.
loaders = {
    role: DataLoader(dataset, batch_size=2, shuffle=(role == "train"), num_workers=0,
                     generator=torch.Generator().manual_seed(42))
    for role, dataset in datasets.items()
}

# 2. 학습 DataLoader에서 첫 배치를 꺼냅니다.
batch = next(iter(loaders["train"]))
# 3. 배치에 담긴 케이스 이름과 항목별 배열 모양을 출력합니다.
print("배치 케이스:", batch["case_id"])
for field in ["coords", "globals", "stress", "displacement", "node_ids"]:
    print(field, tuple(batch[field].shape))

# 4. 입력과 정답의 배열 모양이 케이스 2개/노드 4,096개 설정과 맞는지 검사합니다.
assert batch["coords"].shape == (2, 4096, 3)
assert batch["globals"].shape == (2, 2)
assert batch["stress"].shape == (2, 4096, 1)
assert batch["displacement"].shape == (2, 4096, 3)


### 6.4 배치 정답의 역정규화와 물리량 확인


### 03장 / 셀 24 / Notebook


In [ ]:
# 1. 저장된 학습 기준으로 배치의 응력/변위를 MPa/mm로 역정규화합니다.
stress_mpa = normalizer.denormalize("stress", batch["stress"])
displacement_mm = normalizer.denormalize("displacement", batch["displacement"])
# 2. 선택한 노드 전체의 응력 범위를 출력합니다.
print("배치 응력 범위(MPa):", float(stress_mpa.min()), "~", float(stress_mpa.max()))
# 3. 각 노드의 변위 세 성분으로 이동 거리를 구하고, 가장 큰 값을 출력합니다.
print("선택 노드의 최대변위 크기(mm):", float(displacement_mm.norm(dim=-1).max()))
# 4. 샘플의 최대값을 해석할 때 확인할 점을 출력합니다.
print("샘플 노드의 최대값은 전체 격자의 최대값과 다를 수 있습니다.")


## 7. 학습 자료 준비 상태 점검


### 03장 / 셀 26 / Notebook


In [ ]:
# 1. 공통 모듈로 자료 묶음을 새로 만들어 처리 규칙과 모든 배치의 입력/정답 연결을 점검합니다.
checks = audit_dataset(cases, split, normalizer, fixed_indices)
# 2. 통과 여부와 주요 점검 결과, 역정규화 오차를 출력합니다.
print("전체 점검:", checks["all_passed"])
print("Holdout 변경이 train 통계에 영향 없음:", checks["holdout_mutation_does_not_change_statistics"])
print("겹치는 분할 거부:", checks["overlapping_split_rejected"])
print("노드/정답 대응 확인:", checks["sampled_nodes_unique_and_labels_aligned"])
print("float32 역정규화 최대 절대오차:", checks["float32_inverse_max_absolute_error"])
# 3. 모든 검사를 통과했는지 확인합니다. 실패하면 여기서 멈춥니다.
assert checks["all_passed"]
